# Binary Algorithms Training Notebook
This notebook merges code from dataset.py, model.py, experiment.py, experiment_1.sh, and experiment_2.sh

## 1. Imports and Setup

In [ ]:
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

## 2. Dataset Implementation

In [ ]:
class PermutationDataset(Dataset):
    def __init__(self, P: torch.Tensor):
        """
        Initialize the dataset with a given permutation matrix P.
        Assumes P is a square matrix of shape (n, n).
        """
        super().__init__()
        assert P.dim() == 2 and P.shape[0] == P.shape[1], "P must be a square matrix."
        self.P = P
        self.n = P.shape[0]
        self.basis = torch.eye(self.n, dtype=P.dtype, device=P.device)

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        """
        Return the pair (e_idx, P e_idx):
        - e_idx: standard basis vector with a 1 at position idx.
        - P e_idx: the result of applying the permutation matrix P to e_idx.
        """        
        e_idx = self.basis[idx]
        Pe_idx = self.P @ e_idx

        return e_idx, Pe_idx

def generate_permutation(n, device='cpu'):
    I = torch.eye(n, device=device)
    P = torch.randperm(n, device=device)
    return I[P]

def generate_test(k, num_ones, num_samples, device='cpu'):
    assert num_ones <= k
    x = torch.zeros((num_samples, k), dtype=torch.float32, device=device)
    for xi in x:
        indices = torch.randperm(k)[:num_ones]
        xi[indices] = 1.
    return x

## 3. Model Implementation (NTKMLP)

In [ ]:
class NTKMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, sigma_w = 1.0, sigma_b = 1.0):
        super(NTKMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, output_dim, bias=False)
        self.relu = nn.ReLU()
        self.sigma_w = sigma_w
        self.sigma_b = sigma_b
        self._initialize_weights()

    def _initialize_weights(self):
        """
        Initialize the weights of the network according 
        to the NTK parametrization
        """
        nn.init.normal_(self.fc1.weight, mean=0, std=self.sigma_w / (self.fc1.in_features ** 0.5))
        nn.init.normal_(self.fc1.bias, mean=0, std=self.sigma_b)
        nn.init.normal_(self.fc2.weight, mean=0, std=self.sigma_w / (self.fc2.in_features ** 0.5))
        nn.init.normal_(self.fc2.bias, mean=0, std=self.sigma_b)


    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## 4. Training Functions

In [ ]:
def train_model(train_loader: DataLoader, epochs: int, k: int, hidden_dim: int, sigma_w: float = None, sigma_b: float = None, device: str = 'cpu', lr: float = 1e-3):
    """
    Train a model on the given dataset.
    Args:
    - train_loader: the DataLoader for the training dataset.
    - epochs: the number of epochs to train the model.
    - k: the dimension of the input space.
    - hidden_dim: the dimension of the hidden layer.
    - sigma_w: the standard deviation of the weights initialization.
    - sigma_b: the standard deviation of the bias initialization.
    - device: the device to run the training.
    
    Returns:
    - model: the trained model.
    """
    sigma_w = sigma_w if sigma_w is not None else 1
    sigma_b = sigma_b if sigma_b is not None else 0.0

    model = NTKMLP(k, hidden_dim, k, sigma_w, sigma_b).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = torch.nn.functional.mse_loss

    for epoch in range(epochs):
        for x, y in train_loader:
            optimizer.zero_grad()
            y_pred = model(x)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
        #if epoch % 500 == 0 or epoch == epochs - 1:
        #    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

    return model



def train_and_evaluate(k: int, delta: float, outdir: str, trials: int = 1, epochs: int = 2000, batch_size: int = 256, hidden_dim: int = 2048, num_tests: int = 1000, nnz: int = 2, sigma_w: float = None, sigma_b: float = None, max_models: int = 5_000, device: str = 'cpu', seed: int = 0):
    """
    Train a model on a dataset generated with a random permutation matrix P.
    Evaluate the model on a test set generated with the same permutation matrix P.
    Args:
    - k: the dimension of the input space.
    - delta: the desired error.
    - outdir: the directory to save the results.
    - epochs: the number of epochs to train the model.
    - batch_size: the batch size for training.
    - hidden_dim: the dimension of the hidden layer.
    - num_tests: the number of test samples to evaluate the model.
    - nnz: the number of non-zero elements in the test samples.
    - sigma_w: the standard deviation of the weights initialization.
    - sigma_b: the standard deviation of the bias initialization.
    - device: the device to run the training and evaluation.
    
    Returns:
    - num_models: the number of models trained before reaching the desired error delta.
    """
    for t in range(trials):
        os.makedirs(f"{outdir}/{k}/{t}", exist_ok=True)
        torch.manual_seed(seed+t)
        P = generate_permutation(k, device=device)
        
        train_data = PermutationDataset(P)
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

        X_test = generate_test(k, nnz, num_tests, device)
        Y_test = (P @ X_test.T).T
        X_test = X_test / X_test.norm(p=2, dim=1, keepdim=True)
        Y_test_scaled = Y_test / Y_test.norm(p=2, dim=1, keepdim=True)

        torch.save(P.to("cpu"), f"{outdir}/{k}/{t}/permutation.pt")
        torch.save(X_test.to("cpu"), f"{outdir}/{k}/{t}/test_set_X.pt")
        torch.save(Y_test.to("cpu"), f"{outdir}/{k}/{t}/test_set_Y.pt")

        acc = 0
        error = 1e6
        num_models = 1
        all_preds = None

        while acc < 1 - delta and num_models < max_models:
            model = train_model(train_loader, epochs, k, hidden_dim, sigma_w, sigma_b, device)

            with torch.no_grad():
                pred = model(X_test).unsqueeze(0)
                if all_preds is None:
                    all_preds = pred
                else:
                    all_preds = torch.cat((all_preds, pred), dim=0)

                mean = torch.mean(all_preds, dim=0)
                error = torch.nn.functional.mse_loss(mean, Y_test_scaled)
                error = error if not torch.isnan(error) else 1e6

                mean[mean > 0] = 1.
                mean[mean <= 0] = 0.
                acc = torch.mean((mean.int() == Y_test.int()).all(dim=1).float())

            print(f"Trial {t+1}, Num. models: {num_models} Error: {error:.3f} Acc: {acc:.3f}")
            torch.save(all_preds.to("cpu"), f"{outdir}/{k}/{t}/predictions.pt")

            with open(f"{outdir}/{k}/{t}/results.csv", "a") as f:
                f.write(f"{num_models},{error},{acc}\n")

            num_models += 1

        return num_models

## 5. Run Experiment 1
Configuration from experiment_1.sh: Loop k from 2 to 30

In [ ]:
# Experiment 1 Configuration (from experiment_1.sh)
# for k in {2..30}; do
#     python experiment.py --k $k --outdir results/experiment_1 --trials 1 --hidden_dim 50000 --sigma_w 1 --seed 2 --device cuda --epochs 10000
# done

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

for k in range(2, 16):  # from 2 to 30
    print(f"\n{'='*60}")
    print(f"Running Experiment 1 with k={k}")
    print(f"{'='*60}")
    
    train_and_evaluate(
        k=k,
        delta=0.1,  # default value
        outdir="results/experiment_1",
        trials=1,
        epochs=10000,
        batch_size=256,  # default value
        hidden_dim=50000,
        num_tests=1000,  # default value
        nnz=2,  # default value
        sigma_w=1,
        sigma_b=None,  # default value
        max_models=5000,  # default value
        device=device,
        seed=2
    )

print("\nExperiment 1 completed!")

## 6. Run Experiment 2
Configuration from experiment_2.sh: Loop k in [5, 10, 15, 20, 25, 30] with delta=0.001

In [ ]:
# Experiment 2 Configuration (from experiment_2.sh)
# for k in 5 10 15 20 25 30; do
#     python experiment.py --k $k --outdir results/experiment_2 --trials 1 --hidden_dim 50000 --sigma_w 1 --seed 2 --device cuda --epochs 10000 --delta 0.001
# done

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f"Using device: {device}")

# for k in [5, 10, 15, 20, 25, 30]:
#     print(f"\n{'='*60}")
#     print(f"Running Experiment 2 with k={k}")
#     print(f"{'='*60}")
    
#     train_and_evaluate(
#         k=k,
#         delta=0.001,  # lower delta for experiment 2
#         outdir="results/experiment_2",
#         trials=1,
#         epochs=10000,
#         batch_size=256,  # default value
#         hidden_dim=50000,
#         num_tests=1000,  # default value
#         nnz=2,  # default value
#         sigma_w=1,
#         sigma_b=None,  # default value
#         max_models=5000,  # default value
#         device=device,
#         seed=2
#     )

# print("\nExperiment 2 completed!")